# Feature Engineering — Rossmann Sales Forecasting

## What is Feature Engineering?
Transforming raw data into meaningful input features
that help ML model understand patterns better.

## What we do in this notebook:
1. Load cleaned data from EDA
2. Extract date/time features
3. Create lag features (past sales)
4. Create rolling average features
5. Encode categorical variables
6. Save final feature set for model training

## Why this matters:
Raw data → Model = poor results
Raw data → Feature Engineering → Model = much better results

## Step 1 — Import Libraries

In [1]:
# ── Standard libraries ───────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Display settings ─────────────────────────────────────────────
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✅ Libraries loaded successfully!")

✅ Libraries loaded successfully!


## Step 2 — Load Cleaned Data
Loading the cleaned and merged dataset saved from 01_EDA.ipynb

In [2]:
# ── Load cleaned data from processed folder ───────────────────────
df = pd.read_csv('../data/processed/train_cleaned.csv', 
                 low_memory=False)

# ── Convert Date to datetime ──────────────────────────────────────
df['Date'] = pd.to_datetime(df['Date'])

# ── Filter only open stores ───────────────────────────────────────
# Closed stores have Sales = 0, not useful for forecasting
df = df[df['Open'] == 1].copy()

# ── Sort by Store and Date — critical for lag features ────────────
df = df.sort_values(['Store', 'Date']).reset_index(drop=True)

# ── Verify ────────────────────────────────────────────────────────
print(f"📦 Shape after filtering closed stores : {df.shape}")
print(f"📅 Date range : {df['Date'].min()} → {df['Date'].max()}")
print(f"🏪 Total stores : {df['Store'].nunique()}")
print()
print("✅ Data loaded and ready for feature engineering!")

📦 Shape after filtering closed stores : (844392, 18)
📅 Date range : 2013-01-01 00:00:00 → 2015-07-31 00:00:00
🏪 Total stores : 1115

✅ Data loaded and ready for feature engineering!


## Step 3 — Date & Time Features
Extracting meaningful time-based features from the Date column.
These help the model understand seasonality and time patterns
we discovered during EDA.

In [3]:
# ── Extract date features ─────────────────────────────────────────
df['Year']        = df['Date'].dt.year
df['Month']       = df['Date'].dt.month
df['Day']         = df['Date'].dt.day
df['Week']        = df['Date'].dt.isocalendar().week.astype(int)
df['DayOfYear']   = df['Date'].dt.dayofyear
df['Quarter']     = df['Date'].dt.quarter

# ── Season feature ────────────────────────────────────────────────
# Based on EDA — seasons affect sales significantly
def get_season(month):
    if month in [12, 1, 2]:  return 'Winter'
    elif month in [3, 4, 5]: return 'Spring'
    elif month in [6, 7, 8]: return 'Summer'
    else:                     return 'Autumn'

df['Season'] = df['Month'].apply(get_season)

# ── Is month start/end feature ────────────────────────────────────
df['IsMonthStart'] = df['Date'].dt.is_month_start.astype(int)
df['IsMonthEnd']   = df['Date'].dt.is_month_end.astype(int)

# ── Is weekend feature ────────────────────────────────────────────
df['IsWeekend'] = (df['DayOfWeek'] >= 6).astype(int)

# ── Verify ────────────────────────────────────────────────────────
new_cols = ['Year', 'Month', 'Day', 'Week', 'DayOfYear', 
            'Quarter', 'Season', 'IsMonthStart', 
            'IsMonthEnd', 'IsWeekend']

print("📋 New date features created:")
print("-" * 35)
for col in new_cols:
    print(f"   ✅ {col:<20} — sample: {df[col].iloc[0]}")

print(f"\n📦 Shape now : {df.shape}")

📋 New date features created:
-----------------------------------
   ✅ Year                 — sample: 2013
   ✅ Month                — sample: 1
   ✅ Day                  — sample: 2
   ✅ Week                 — sample: 1
   ✅ DayOfYear            — sample: 2
   ✅ Quarter              — sample: 1
   ✅ Season               — sample: Winter
   ✅ IsMonthStart         — sample: 0
   ✅ IsMonthEnd           — sample: 0
   ✅ IsWeekend            — sample: 0

📦 Shape now : (844392, 28)


## Step 4 — Lag Features
Lag features = past sales values as input features.
These are the most powerful features for sales forecasting.

Example:
- Lag_7  = sales from 7 days ago (same day last week)
- Lag_14 = sales from 14 days ago
- Lag_28 = sales from 28 days ago (same day 4 weeks ago)

Why they work:
→ Yesterday's sales predict today's sales
→ Same weekday last week is strong signal
→ Model learns "if last week was high, this week likely high too"

In [4]:
# ── Create lag features per store ─────────────────────────────────
# IMPORTANT: groupby Store — lags must be within same store!
# Otherwise Store 1's sales leak into Store 2's features

lag_days = [7, 14, 21, 28]

for lag in lag_days:
    df[f'Lag_{lag}'] = df.groupby('Store')['Sales'].shift(lag)
    print(f"✅ Lag_{lag:<3} created")

print()
print(f"📦 Shape now : {df.shape}")
print()

# ── Preview lag features ──────────────────────────────────────────
print("📋 Sample — Store 1 first few rows with lags:")
sample = df[df['Store'] == 1][
    ['Date', 'Sales', 'Lag_7', 'Lag_14', 'Lag_21', 'Lag_28']
].head(35).tail(5)
print(sample.to_string())

✅ Lag_7   created
✅ Lag_14  created
✅ Lag_21  created
✅ Lag_28  created

📦 Shape now : (844392, 32)

📋 Sample — Store 1 first few rows with lags:
         Date  Sales   Lag_7  Lag_14  Lag_21  Lag_28
30 2013-02-06   6140 3725.00 5394.00 4952.00 4486.00
31 2013-02-07   5499 4601.00 5720.00 4717.00 4997.00
32 2013-02-08   5681 4709.00 5578.00 3900.00 7176.00
33 2013-02-09   5370 5633.00 5195.00 4008.00 5580.00
34 2013-02-11   4409 5970.00 5586.00 4044.00 5471.00


## Step 5 — Rolling Average Features
Rolling averages smooth out daily fluctuations and capture
the recent sales trend for each store.

Example:
- Rolling_7  = average sales over last 7 days
- Rolling_14 = average sales over last 14 days
- Rolling_28 = average sales over last 28 days

Why they work:
→ Captures recent sales momentum
→ Smooths out one-off spikes or dips
→ Tells model "is this store trending up or down lately?"

In [5]:
# ── Rolling average features per store ───────────────────────────
# min_periods=1 means calculate even if fewer days available
windows = [7, 14, 28]

for window in windows:
    df[f'Rolling_Mean_{window}'] = (
        df.groupby('Store')['Sales']
        .transform(lambda x: x.shift(1).rolling(
            window=window, min_periods=1).mean())
    )
    df[f'Rolling_Std_{window}'] = (
        df.groupby('Store')['Sales']
        .transform(lambda x: x.shift(1).rolling(
            window=window, min_periods=1).std())
    )
    print(f"✅ Rolling_Mean_{window} & Rolling_Std_{window} created")

print()
print(f"📦 Shape now : {df.shape}")
print()

# ── Preview rolling features ──────────────────────────────────────
print("📋 Sample — Store 1 rolling features:")
sample = df[df['Store'] == 1][
    ['Date', 'Sales', 'Rolling_Mean_7', 
     'Rolling_Mean_14', 'Rolling_Mean_28']
].head(35).tail(5)
print(sample.to_string())

✅ Rolling_Mean_7 & Rolling_Std_7 created
✅ Rolling_Mean_14 & Rolling_Std_14 created
✅ Rolling_Mean_28 & Rolling_Std_28 created

📦 Shape now : (844392, 38)

📋 Sample — Store 1 rolling features:
         Date  Sales  Rolling_Mean_7  Rolling_Mean_14  Rolling_Mean_28
30 2013-02-06   6140         5388.43          5346.07          5116.36
31 2013-02-07   5499         5733.43          5399.36          5175.43
32 2013-02-08   5681         5861.71          5383.57          5193.36
33 2013-02-09   5370         6000.57          5390.93          5139.96
34 2013-02-11   4409         5963.00          5403.43          5132.46


## Step 6 — Encode Categorical Variables
ML models only understand numbers — not text.
We need to convert text columns to numerical format.

Columns to encode:
- StoreType      : a, b, c, d → 0, 1, 2, 3
- Assortment     : a, b, c → 0, 1, 2
- StateHoliday   : 0, a, b, c → 0, 1, 2, 3
- Season         : Winter, Spring, Summer, Autumn → 0,1,2,3
- PromoInterval  : None, Jan,Apr... → numerical

In [6]:
from sklearn.preprocessing import LabelEncoder

# ── Label encode categorical columns ─────────────────────────────
cat_cols = ['StoreType', 'Assortment', 'StateHoliday', 
            'Season', 'PromoInterval']

le = LabelEncoder()

for col in cat_cols:
    df[col] = df[col].astype(str)  # ensure string type
    df[col] = le.fit_transform(df[col])
    print(f"✅ {col:<20} encoded — unique values: {df[col].nunique()}")

print()
print(f"📦 Shape now : {df.shape}")
print()

# ── Verify all columns are now numeric ───────────────────────────
non_numeric = df.select_dtypes(include=['object']).columns.tolist()
if len(non_numeric) == 0:
    print("✅ All columns are now numeric — ready for ML!")
else:
    print(f"⚠️  Non-numeric columns remaining: {non_numeric}")

✅ StoreType            encoded — unique values: 4
✅ Assortment           encoded — unique values: 3
✅ StateHoliday         encoded — unique values: 4
✅ Season               encoded — unique values: 4
✅ PromoInterval        encoded — unique values: 4

📦 Shape now : (844392, 38)

✅ All columns are now numeric — ready for ML!


## Step 7 — Handle NaN from Lag Features & Final Cleanup
Lag features create NaN for first few rows of each store
(no past data available yet). We need to handle these
before saving the final dataset.

In [7]:
# ── Check NaN created by lag/rolling features ─────────────────────
print("📋 NaN count per lag/rolling feature:")
print("-" * 45)
lag_roll_cols = [c for c in df.columns if 
                 'Lag_' in c or 'Rolling_' in c]
for col in lag_roll_cols:
    null_count = df[col].isnull().sum()
    print(f"   {col:<25} : {null_count:,} NaN")

print()

# ── Drop rows where lag features are NaN ─────────────────────────
# First 28 rows of each store will have NaN lags
df_clean = df.dropna(subset=lag_roll_cols).copy()

print(f"📦 Shape before dropna : {df.shape}")
print(f"📦 Shape after dropna  : {df_clean.shape}")
print(f"🗑️  Rows removed        : {df.shape[0] - df_clean.shape[0]:,}")
print()

# ── Final NaN check ───────────────────────────────────────────────
total_nan = df_clean.isnull().sum().sum()
print(f"✅ Total NaN remaining : {total_nan}")
print()
print("✅ Data is clean and ready to save!")

📋 NaN count per lag/rolling feature:
---------------------------------------------
   Lag_7                     : 7,805 NaN
   Lag_14                    : 15,610 NaN
   Lag_21                    : 23,415 NaN
   Lag_28                    : 31,220 NaN
   Rolling_Mean_7            : 1,115 NaN
   Rolling_Std_7             : 2,230 NaN
   Rolling_Mean_14           : 1,115 NaN
   Rolling_Std_14            : 2,230 NaN
   Rolling_Mean_28           : 1,115 NaN
   Rolling_Std_28            : 2,230 NaN

📦 Shape before dropna : (844392, 38)
📦 Shape after dropna  : (813172, 38)
🗑️  Rows removed        : 31,220

✅ Total NaN remaining : 0

✅ Data is clean and ready to save!


## Step 8 — Select Final Features & Save
Selecting the most relevant features for ML model
and saving the final dataset for model training.

In [8]:
# ── Define final feature columns ──────────────────────────────────
feature_cols = [
    # Target
    'Sales',

    # Store info
    'Store', 'StoreType', 'Assortment',
    'CompetitionDistance', 'CompetitionOpenSinceMonth',
    'CompetitionOpenSinceYear',

    # Promo
    'Promo', 'Promo2', 'Promo2SinceWeek',
    'Promo2SinceYear', 'PromoInterval',

    # Date features
    'Year', 'Month', 'Day', 'Week',
    'DayOfWeek', 'DayOfYear', 'Quarter',
    'Season', 'IsWeekend', 'IsMonthStart', 'IsMonthEnd',

    # Holiday
    'StateHoliday', 'SchoolHoliday',

    # Lag features
    'Lag_7', 'Lag_14', 'Lag_21', 'Lag_28',

    # Rolling features
    'Rolling_Mean_7', 'Rolling_Mean_14', 'Rolling_Mean_28',
    'Rolling_Std_7',  'Rolling_Std_14',  'Rolling_Std_28',
]

# ── Select final columns ──────────────────────────────────────────
df_final = df_clean[feature_cols].copy()

# ── Save to processed folder ──────────────────────────────────────
df_final.to_csv('../data/processed/train_features.csv', index=False)

# ── Verify ────────────────────────────────────────────────────────
import os
file_size = os.path.getsize(
    '../data/processed/train_features.csv') / (1024 * 1024)

print(f"📦 Final dataset shape  : {df_final.shape}")
print(f"📋 Total features       : {df_final.shape[1] - 1} + 1 target")
print(f"📁 File size            : {file_size:.2f} MB")
print()
print("📋 Feature Summary:")
print("-" * 40)
print(f"   Store features       : 6")
print(f"   Promo features       : 5")
print(f"   Date/Time features   : 10")
print(f"   Holiday features     : 2")
print(f"   Lag features         : 4")
print(f"   Rolling features     : 6")
print(f"   {'─'*25}")
print(f"   Total features       : 33")
print()
print("🎉 Feature Engineering COMPLETE!")

📦 Final dataset shape  : (813172, 35)
📋 Total features       : 34 + 1 target
📁 File size            : 165.38 MB

📋 Feature Summary:
----------------------------------------
   Store features       : 6
   Promo features       : 5
   Date/Time features   : 10
   Holiday features     : 2
   Lag features         : 4
   Rolling features     : 6
   ─────────────────────────
   Total features       : 33

🎉 Feature Engineering COMPLETE!
